# 🟡 Lesson 08 — PyProj

**Level: Intermediate** · Coordinate reference systems and transformations — convert GPS lat/lon to UTM metres correctly.

*Part of the companion package for [python_for_geologists](https://github.com/kevinalexandr19/python_for_geologists) by Kevin Alexander Gomez.*

In [1]:
from pyproj import CRS, Transformer, Geod
import pyproj
print("pyproj", pyproj.__version__)

pyproj 3.7.2


## 1. Inspect a CRS

In [2]:
utm18s = CRS.from_epsg(32718)
print(utm18s.name)
print("units:", utm18s.axis_info[0].unit_name)
print("area of use:", utm18s.area_of_use.name)

WGS 84 / UTM zone 18S
units: metre
area of use: Between 78°W and 72°W, southern hemisphere between 80°S and equator, onshore and offshore. Argentina. Brazil. Chile. Colombia. Ecuador. Peru.


## 2. Transform lat/lon → UTM
`always_xy=True` keeps the (lon, lat) / (easting, northing) order sane.

In [3]:
t = Transformer.from_crs("EPSG:4326", "EPSG:32718", always_xy=True)

# Misti volcano, Arequipa
lon, lat = -71.409, -16.294
e, n = t.transform(lon, lat)
print(f"Misti: lon {lon}, lat {lat}  ->  E {e:,.0f} m, N {n:,.0f} m (UTM 18S)")

# and back again
lon2, lat2 = t.transform(e, n, direction="INVERSE")
print(f"round trip: {lon2:.4f}, {lat2:.4f}")

Misti: lon -71.409, lat -16.294  ->  E 883,853 m, N 8,195,165 m (UTM 18S)
round trip: -71.4090, -16.2940


## 3. Batch transform — works on arrays too

In [4]:
import numpy as np
lons = np.array([-77.03, -71.54, -70.02])   # Lima, Arequipa, Tacna
lats = np.array([-12.05, -16.41, -18.01])
es, ns = t.transform(lons, lats)
for city, e_, n_ in zip(["Lima", "Arequipa", "Tacna"], es, ns):
    print(f"{city:<9} E {e_:>12,.0f}  N {n_:>13,.0f}")

Lima      E      279,014  N     8,667,100
Arequipa  E      869,617  N     8,182,556
Tacna     E    1,027,707  N     8,001,605


## 4. Geodesic distances — true distance on the ellipsoid
Never compute distances in degrees!

In [5]:
g = Geod(ellps="WGS84")

# Lima -> Arequipa
az12, az21, dist = g.inv(-77.03, -12.05, -71.54, -16.41)
print(f"Lima -> Arequipa: {dist/1000:.1f} km, azimuth {az12:.1f} deg")

# area of a polygon drawn on the ellipsoid (a rough claim block)
lons_p = [-71.6, -71.4, -71.4, -71.6]
lats_p = [-16.5, -16.5, -16.3, -16.3]
area, perim = g.polygon_area_perimeter(lons_p, lats_p)
print(f"block area: {abs(area)/1e6:.1f} km2, perimeter {perim/1000:.1f} km")

Lima -> Arequipa: 763.9 km, azimuth 129.8 deg
block area: 472.8 km2, perimeter 87.0 km


## 5. Which UTM zone am I in?

In [6]:
from pyproj.aoi import AreaOfInterest
from pyproj.database import query_utm_crs_info

info = query_utm_crs_info(
    datum_name="WGS 84",
    area_of_interest=AreaOfInterest(west_lon_degree=-71.5, south_lat_degree=-16.5,
                                    east_lon_degree=-71.3, north_lat_degree=-16.2))
for i in info:
    print(i.code, i.name)

32719 WGS 84 / UTM zone 19S


### ✏️ Try it
1. Transform the drillhole collar (E 500000, N 8600000, UTM 18S) back to lat/lon.
2. Use `Geod.npts()` to generate 10 waypoints along the Lima→Arequipa line.

📚 Docs: https://pyproj4.github.io/pyproj/stable/